# Topic: Error Handling Security - Traceback information disclosures, private logging, and sanitised generic errors
 
## 1. THE INFORMATION DISCLOSURE RISK
- **Information Disclosure via Exceptions**:
  - Returning raw tracebacks or detailed database exception strings (like `sqlite3.OperationalError: no such column: pass`) to clients is a major vulnerability.
  - It reveals internal system paths, framework versions, database schema names, and library dependencies, enabling attackers to map the application's attack surface.
 
## 2. SECURE EXCEPTION PATTERN
- **The Practice**:
  1. **Catch specific exceptions**: Do not let errors propagate unhandled to the user interface.
  2. **Log privately**: Write full tracebacks and debug states to secure, private server logs for developers.
  3. **Return generic warnings**: Show end-users a generic, friendly message (e.g., "Internal Server Error").
  4. **Track with Correlation IDs**: Generate a unique tracking identifier (Incident ID) for each error. Log the ID with the traceback, and show the same ID to the user. This allows developers to locate the trace quickly in the logs without leaking details to users.
 
---


In [ ]:
import traceback
import uuid

# Mock private server log storage
private_server_logs = {}

# 1. Vulnerable API Endpoint returning raw tracebacks
def vulnerable_api_get_user(user_id):
    """Vulnerable implementation returning traceback dumps."""
    try:
        # Simulate database index error
        users_list = ["Alice", "Bob"]
        # If user_id is out of range, index error is thrown
        username = users_list[int(user_id)]
        return {"status": "success", "username": username}
    except Exception as e:
        # DANGEROUS: Returning raw traceback details to user
        tb_str = traceback.format_exc()
        return {
            "status": "error",
            "message": f"Exception raised: {str(e)}",
            "traceback": tb_str
        }

print("--- Vulnerable API Output (Invalid ID) ---")
err_output = vulnerable_api_get_user(99)
print(f"Status: {err_output['status']}")
print(f"Message: {err_output['message']}")
print("Traceback returned to user:")
print(err_output['traceback'])
# Expected: User sees complete folder structures, file lines, and index out-of-range details!



In [ ]:
# 2. Secure API Endpoint returning sanitized error messages
def secure_api_get_user(user_id):
    """Secure implementation sanitizing exceptions."""
    try:
        users_list = ["Alice", "Bob"]
        username = users_list[int(user_id)]
        return {"status": "success", "username": username}
    except Exception as e:
        # 1. Generate unique Incident tracking ID
        incident_id = str(uuid.uuid4())
        
        # 2. Save full traceback details to private logs mapped by Incident ID
        tb_str = traceback.format_exc()
        private_server_logs[incident_id] = {
            "error_message": str(e),
            "traceback": tb_str,
            "params": {"user_id": user_id}
        }
        
        # 3. Return a sanitised, generic warning with tracking ID
        return {
            "status": "error",
            "message": "An internal server error occurred. Please contact support.",
            "incident_id": incident_id
        }

print("\n--- Secure API Output (Invalid ID) ---")
secure_output = secure_api_get_user(99)
print(f"Status:      {secure_output['status']}")
print(f"Message:     {secure_output['message']}")
print(f"Incident ID: {secure_output['incident_id']}")
# Expected: User sees no tracebacks, only a clean, generic message and an incident tracking identifier.



In [ ]:
# 3. Developer Audit lookups in Private logs using Incident ID
print("\n--- Developer Private Log Audit ---")
lookup_id = secure_output['incident_id']
if lookup_id in private_server_logs:
    log_entry = private_server_logs[lookup_id]
    print(f"Auditing Incident: {lookup_id}")
    print(f"Error Message:     {log_entry['error_message']}")
    print(f"Parameters passed: {log_entry['params']}")
    print("Full Private Traceback:")
    print(log_entry['traceback'])
